# LingoSync: fine-tune Whisper-small on a free Colab GPU

**Before you start:** Runtime > Change runtime type > **T4 GPU** > Save. Then Runtime > **Run all**.

When asked, choose your **kaggle.json** file (copied to your Desktop). Keep this tab open until the last cell prints DONE (about 1.5-3 hours).

What happens:
1. Downloads the prepared training data from your private Kaggle dataset (hemavathanypetarasu/lingosync-data).
2. LoRA fine-tunes openai/whisper-small (3 passes over ~25 h of Malay-English and Tamil speech).
3. Uploads the fine-tuned model to a new private Kaggle dataset (hemavathanypetarasu/lingosync-whisper-small-model), so it can be downloaded to the laptop automatically.

In [ ]:
# 1. Check that a GPU is attached
import subprocess
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], capture_output=True, text=True)
print(gpu.stdout or gpu.stderr)
assert gpu.returncode == 0, 'No GPU. Use Runtime > Change runtime type > T4 GPU, then Run all again.'

In [ ]:
# 2. Install the same library versions that were tested on the laptop
!pip -q install "transformers==4.46.3" "peft==0.13.2" "accelerate==1.1.1" jiwer kaggle

In [ ]:
# 3. Choose your kaggle.json file when the button appears
import os
from google.colab import files
uploaded = files.upload()
assert uploaded, 'Please choose kaggle.json'
os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'wb') as handle:
    handle.write(next(iter(uploaded.values())))
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('Kaggle key installed')

In [ ]:
# 4. Download the training data from your private Kaggle dataset
!kaggle datasets download -d hemavathanypetarasu/lingosync-data -p /content/ds --unzip -q
!ls /content/ds /content/ds/lingosync_data/data/splits

In [ ]:
# 5. Train (progress is printed every 10 steps; about 1.5-3 hours on a T4)
!python /content/ds/finetune_lora.py --gpu --base openai/whisper-small --data-root /content/ds/lingosync_data --output /content/whisper-small-lingosync --max-steps 1750 --batch-size 16 --save-steps 250 --validation-clips 300

In [ ]:
# 6. Upload the fine-tuned model to a private Kaggle dataset
import json
model_dir = '/content/whisper-small-lingosync'
with open(f'{model_dir}/dataset-metadata.json', 'w') as handle:
    json.dump({'title': 'LingoSync Whisper small model', 'id': 'hemavathanypetarasu/lingosync-whisper-small-model',
               'licenses': [{'name': 'other'}]}, handle)
!kaggle datasets create -p /content/whisper-small-lingosync -q
print(open(f'{model_dir}/training_info.json').read()[:600])
print('DONE - tell Claude the training has finished.')